#### Download Nvidia Annual Report

In [1]:
import os
import requests
from pathlib import Path


# Create data directory if it doesn't exist
data_folder = Path('..') / 'data'
os.makedirs(data_folder, exist_ok=True)

# Download the PDF file
url = 'https://s201.q4cdn.com/141608511/files/doc_financials/2025/annual/NVIDIA-2025-Annual-Report.pdf'
response = requests.get(url)

# Save the file to data directory
file_path = data_folder / 'NVIDIA-2025-Annual-Report.pdf'
with open(file_path, 'wb') as f:
    f.write(response.content)

print(f"File downloaded and saved to {file_path}")

File downloaded and saved to ..\data\NVIDIA-2025-Annual-Report.pdf


#### Convert to markdown for chunking

In [2]:
from markitdown import MarkItDown

# Initialize MarkItDown
converter = MarkItDown()

# Convert PDF to markdown
markdown_content = converter.convert(file_path)

#### Use simple langchain chunking

In [3]:
markdown_content = markdown_content.text_content

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Create a RecursiveCharacterTextSplitter with 384 max characters per chunk
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len,
)

chunks = text_splitter.create_documents([markdown_content])

c:\Users\neutr\.conda\envs\graph_rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Call the graphrag

In [5]:
import sys

# Add parent directory to path
parent_dir = os.path.dirname(os.getcwd())
sys.path.append(parent_dir)

In [6]:
from src.graph_rag import GLiNERGraphStore

In [37]:
graph_store = GLiNERGraphStore(
    model_path = r"D:\Projects\graph_rag\models\knowledgator--gliner-relex-large-v1.0",
    collection_name="test_collection",
    labels=['organization'],
    relations=['supplies to', 'competitor of', 'supplied by']
)


In [38]:
enriched = graph_store.add_documents(chunks)

100%|██████████| 953/953 [07:01<00:00,  2.26it/s]


In [39]:
# graph_store.edge_df

In [40]:
graph_store.edge_df.query('graph_type == "TRIPLET"')['head'].unique()

<StringArray>
[                  'nvidia corporation',
                               'nvidia',
                            'blackwell',
 'financials and strategic positioning',
                                 'cpus',
                                'grace',
                           'processors',
                             'spectrum',
                                'rubin',
                                 'tsmc',
 ...
                        'u.s. treasury',
                            'dgx cloud',
                                  'gpu',
                              'federal',
               'compute and networking',
                             'graphics',
                    'direct customer a',
    'computershare trust company, n.a.',
                        'computershare',
                           'cooley llp']
Length: 257, dtype: str

In [41]:
# graph_store.edge_df.query()

In [47]:
graph_store.graph_triplet

In [48]:
def get_nodes_at_n_hops(G, source, n):
    # Get shortest path lengths from source to all reachable nodes
    path_lengths = nx.single_source_shortest_path_length(G, source, cutoff=n)
    
    # Filter for nodes where the distance is exactly n
    nodes_at_n = [node for node, length in path_lengths.items() if length == n]
    
    return nodes_at_n

In [49]:
import networkx as nx

In [50]:
import networkx as nx

def get_edges_within_n_hops(G, source_node, n, data=True):
    # 1. Get all nodes at distance <= n
    # This is highly efficient as it stops searching once the cutoff is reached
    lengths = nx.single_source_shortest_path_length(G, source_node, cutoff=n)
    
    # 2. Induce a subgraph view
    # Subgraph views in NetworkX are memory efficient references to the original G
    subgraph_view = G.subgraph(lengths.keys())
    
    # 3. Return the edges within this neighborhood
    return list(subgraph_view.edges(data=data))

In [ ]:
import networkx as nx

def get_outgoing_edges_within_n_hops(G, source, n):
    lengths = nx.single_source_shortest_path_length(G, source, cutoff=n)
    result_edges = []
    
    for node in lengths:
        if lengths[node] < n:
            # Instead of passing 'data=True', we use 'keys=True, data=True' 
            # and manually iterate over the MultiDiGraph edges.
            # This avoids the internal reportview lookup that causes the TypeError.
            for u, v, key, data in G.out_edges(node, keys=True, data=True):
                result_edges.append((u, v, data))
            
    return result_edges

In [60]:
G = graph_store.graph_triplet
source = 'tsmc'
n = 1

In [ ]:
import networkx as nx

def get_outgoing_edges_within_n_hops(G, source, n):
    lengths = nx.single_source_shortest_path_length(G, source, cutoff=n)
    result_edges = []
    
    for node in lengths:
        if lengths[node] < n:
            # Instead of passing 'data=True', we use 'keys=True, data=True' 
            # and manually iterate over the MultiDiGraph edges.
            # This avoids the internal reportview lookup that causes the TypeError.
            for u, v, key, data in G.out_edges(node, keys=True, data=True):
                result_edges.append((u, v, data))
            
    return result_edges

TypeError: unhashable type: 'dict'

In [53]:
lengths = nx.single_source_shortest_path_length(graph_store.graph_triplet, 'tsmc', cutoff=1)

In [55]:
subgraph_view = graph_store.graph_triplet.subgraph(lengths.keys())

In [58]:
# Assuming 'edges' is the output of your function
for u, v, data in subgraph_view.edges(data=True):
    print(f"Source: {u} | Target: {v} | Relation: {data.get('relation')} | SourceID: {data.get('edge_source')}")

Source: we | Target: we | Relation: competitor of | SourceID: faa45d1f-a4f9-4948-a4ae-1ab819ed37b8
Source: we | Target: we | Relation: competitor of | SourceID: e702bb67-4e9e-4ec1-b460-e0068a31682a
Source: we | Target: we | Relation: competitor of | SourceID: 58d5a176-3f8c-4f8d-816c-0ddd8c14419d
Source: we | Target: we | Relation: competitor of | SourceID: b6759dbb-2140-43d0-8442-e0343bf93fae
Source: we | Target: we | Relation: competitor of | SourceID: 0053c964-7135-41a6-8f32-f0908a175d2d
Source: we | Target: we | Relation: competitor of | SourceID: ae2895e9-3021-4d32-b7f3-cf073b2e7603
Source: we | Target: we | Relation: competitor of | SourceID: bf2443e0-ead9-4711-96c4-c34ea6dc9429
Source: we | Target: we | Relation: competitor of | SourceID: 0d221753-c55f-41bd-9869-57dfe4b0b866
Source: we | Target: we | Relation: competitor of | SourceID: 794e4473-7a38-48d3-8f1f-b52fae342094
Source: we | Target: we | Relation: competitor of | SourceID: f14144e7-0ae0-4589-9fcb-9d251523993e
Source: we